# 04 — Neural avalanches

**Goal.** Reproduce the Beggs-Plenz 2003 / Plenz-Thiagarajan 2007 result: neural avalanches in cortical tissue follow a power-law size distribution with τ ≈ 1.5 (slice) to 2.5 (in-vivo / network-level).

**ETA.** ~3 minutes.

**Source.** Synthetic Pareto with τ=2.58 (matches structural-isomorphism Phase 8 mouse-cortex result). Real LFP / spike data can be plugged in via `event_sizes = np.loadtxt('your_avalanches.txt')`.

**Expected.** τ ∈ [2.3, 2.8].

In [ ]:
# 1. Synthetic avalanche-size sample (τ=2.58)
import numpy as np

rng = np.random.default_rng(7)
tau_true = 2.58
u = rng.uniform(0, 1, 6000)
sizes = (1.0 - u) ** (-1.0 / (tau_true - 1.0))
# Round to integers (discrete event counts)
sizes = np.ceil(sizes).astype(float)
print(f"n = {len(sizes)}, max avalanche size = {int(sizes.max())}")

In [ ]:
# 2. Discrete Clauset MLE
from soc_pipeline import fit_clauset_powerlaw, bootstrap_ci

fit = fit_clauset_powerlaw(sizes, discrete=True, name="neural_avalanche")
print(f"τ = {fit.alpha:.3f} ± {fit.sigma:.3f}")
print(f"xmin = {fit.xmin:.0f}")
print(f"n_tail = {fit.n_tail}")
print(f"Vuong vs lognormal: R = {fit.vs_lognormal_R}")
print(f"Vuong vs exponential: R = {fit.vs_exponential_R}")

ci = bootstrap_ci(sizes, n_boot=40, discrete=True, seed=7)
print(f"\n95% bootstrap CI: [{ci.ci_low:.3f}, {ci.ci_high:.3f}]")

In [ ]:
# 3. CCDF plot
import matplotlib.pyplot as plt
from soc_pipeline import empirical_ccdf

grid, ccdf = empirical_ccdf(sizes, n_points=120)
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(grid, ccdf, "o", markersize=4, alpha=0.6, label="empirical")
tg = grid[grid >= fit.xmin]
tcc = (tg / fit.xmin) ** -(fit.alpha - 1) * (sizes >= fit.xmin).mean()
ax.loglog(tg, tcc, "r-", label=f"τ = {fit.alpha:.2f}")
ax.set_xlabel("avalanche size s (electrodes)")
ax.set_ylabel("P(S > s)")
ax.set_title("Neural avalanche size CCDF")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Verdict
from soc_pipeline import verdict_from_alpha_band

v = verdict_from_alpha_band(fit.alpha, predicted=(2.3, 2.8), literature=(1.5, 3.0))
print(f"verdict: {v}")
print(f"Paper headline: τ ≈ 2.58 (mouse cortex, structural-isomorphism Phase 8)")
print(f"Beggs-Plenz 2003 slice value: τ ≈ 1.5 (different observable, smaller spatial scale)")